In [27]:
from transformers import XLMRobertaTokenizer

# Path to your vocabulary file
vocab_file = "pretrained_model/bert_module_khmer/assets/vocab_file_BERT_ copy.txt"

# Initialize a tokenizer instance with XLMRobertaTokenizer
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

# Load vocabulary from file and assign IDs dynamically
with open(vocab_file, "r", encoding="utf-8") as f:
    vocab = [line.strip() for line in f]

# Add special tokens manually
special_tokens = ["<unk>", "<s>", "</s>", "<pad>"]
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

# Add remaining tokens from vocabulary
tokenizer.add_tokens(vocab)

# Save tokenizer model in the format compatible with transformers
tokenizer.save_pretrained("khmer_tokenizer")



('khmer_tokenizer/tokenizer_config.json',
 'khmer_tokenizer/special_tokens_map.json',
 'khmer_tokenizer/sentencepiece.bpe.model',
 'khmer_tokenizer/added_tokens.json')

In [28]:
import os
import torch
import evaluate
import numpy as np
import pandas as pd
import glob as glob
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

from PIL import Image
from zipfile import ZipFile
from tqdm.notebook import tqdm
from dataclasses import dataclass
from torch.utils.data import Dataset
from urllib.request import urlretrieve
from transformers import (
    VisionEncoderDecoderModel, 
    TrOCRProcessor,
    AutoTokenizer,
    ViTFeatureExtractor,
    BertTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator
)

In [23]:
from transformers import XLMRobertaTokenizer

# Load the tokenizer from the saved directory
tokenizer = XLMRobertaTokenizer.from_pretrained("khmer_tokenizer")

# Test tokenizer with a simple input
text = "អាហ៊ូណាន់"
encoded = tokenizer.encode(text)

# Check the type of encoded object
print(type(encoded))

# Print the encoded object to inspect its content
print(encoded)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


<class 'list'>
[0, 250002, 2]


In [25]:
decode=tokenizer.decode(encoded)
print(decode)

<s> អាហ៊ូណាន់ </s>


In [29]:
import os
import pandas as pd

class DatasetConfig:
    DATA_ROOT = "dataset/"

# Load train and test data
train_df = pd.read_fwf(
    os.path.join(DatasetConfig.DATA_ROOT, 'data_train.txt'), header=None
)
train_df.rename(columns={0: 'file_name', 1: 'text'}, inplace=True)

test_df = pd.read_fwf(
    os.path.join(DatasetConfig.DATA_ROOT, 'data_test.txt'), header=None
)
test_df.rename(columns={0: 'file_name', 1: 'text'}, inplace=True)


In [1]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers
from transformers import PreTrainedTokenizerFast
from PIL import Image
import torch
from torch.utils.data import Dataset
import pandas as pd

# Example of how to train a BPE tokenizer on Khmer text (if needed)
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
tokenizer.decoder = decoders.ByteLevel()

# Assuming you have a diverse Khmer text corpus in a file
khmer_corpus_file = "dataset/unique_words.txt"

trainer = trainers.BpeTrainer(
    vocab_size=30000,  # Adjust as needed
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
)
tokenizer.train(files=[khmer_corpus_file], trainer=trainer)

# Save the tokenizer
tokenizer.save("khmer_tokenizer.json")

# Load the tokenizer with PreTrainedTokenizerFast for fast processing
khmer_tokenizer = PreTrainedTokenizerFast(tokenizer_file="khmer_tokenizer.json")


In [2]:
class KhmerDataset(Dataset):
    def __init__(self, root_dir, df, tokenizer, max_target_length=128):
        self.root_dir = root_dir
        self.df = df
        self.tokenizer = tokenizer
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_name = self.df['file_name'].iloc[idx]
        text = self.df['text'].iloc[idx]

        # Read the image
        image = Image.open(f"{self.root_dir}/{file_name}").convert('RGB')

        # Tokenize the text
        tokenized_output = self.tokenizer(
            text,
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True
        )
        labels = tokenized_output.input_ids

        # Replace padding token IDs with -100
        labels = [label if label != self.tokenizer.pad_token_id else -100 for label in labels]

        # Ensure labels are not empty
        if all(label == -100 for label in labels):
            labels = [-100] * self.max_target_length

        # Convert labels to tensor
        labels = torch.tensor(labels)

        # Example: You may want to apply augmentations here on the image
        # image = train_transforms(image)

        # Process the image and get pixel values
        pixel_values = self._process_image(image)

        return {"pixel_values": pixel_values, "labels": labels}

    def _process_image(self, image):
        # Example: You can customize this function for your specific image processing needs
        return self.tokenizer(images=image, return_tensors='pt').pixel_values.squeeze()


: 

In [ ]:
# Example DataFrame
data = {
    'file_name': ['image1.png', 'image2.png'],
    'text': ['សួស្វាគមន៍ស្តង់ដា', 'ខ្មែរជាភាសាជាតិ']
}
df = pd.DataFrame(data)

# Initialize the dataset
khmer_dataset = KhmerDataset(root_dir="/path/to/images", df=df, tokenizer=khmer_tokenizer)

# Example: Fetch an item from the dataset
item = khmer_dataset[0]
print(item)


In [30]:
class KhmerDataset(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=128):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # The image file name.
        file_name = self.df['file_name'][idx]
        # The text (label).
        text = self.df['text'][idx]
        # Read the image, apply augmentations, and get the transformed pixels.
        image = Image.open(self.root_dir + file_name).convert('RGB')
        # image = train_transforms(image)
        pixel_values = self.processor(image, return_tensors='pt').pixel_values
        # Pass the text through the tokenizer and get the labels,
        # i.e. tokenized labels.
        labels = self.processor.tokenizer(
            text,
            padding='max_length',
            max_length=self.max_target_length
        ).input_ids
        # We are using -100 as the padding token.
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]
        # Check if text is empty after tokenization
        if len(labels) == 0 or all(label == -100 for label in labels):
            # Replace with a placeholder or handle as needed
            labels = [self.processor.tokenizer.pad_token_id]

        encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}
        return encoding
        

In [32]:
# from transformers import XLMRobertaTokenizer

# # Specify the directory where your tokenizer files are located
# tokenizer_directory = "khmer_tokenizer/khmer.model"

# # Initialize the tokenizer
# tokenizer = XLMRobertaTokenizer.from_pretrained(tokenizer_directory)

# # Example usage
# text = "សួស្តីអ្នកទាំងអស់គ្នា"
# tokens = tokenizer.tokenize(text)
# token_ids = tokenizer.convert_tokens_to_ids(tokens)

# print("Tokens:", tokens)
# print("Token IDs:", token_ids)


Tokens: ['▁សួស្តី', 'អ្នក', 'ទាំងអស់', 'គ្នា']
Token IDs: [6980, 2677, 8256, 434]


/home/EmHengly/miniconda3/envs/khocr/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2004: FutureWarning: Calling XLMRobertaTokenizer.from_pretrained() with the path to a single file or url is deprecated and won't be possible anymore in v5. Use a model identifier or the path to a directory instead.
  warnings.warn(


In [31]:
# from transformers import VisionEncoderDecoderModel, EncoderDecoderConfig, Seq2SeqTrainer, Seq2SeqTrainingArguments,TrOCRProcessor
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.utils.data import DataLoader

# Assuming you have loaded your model and processor as before
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-small-printed')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-small-printed')

# Check if the model configuration has decoder_start_token_id
# if not hasattr(model.config, "decoder_start_token_id"):
#     # Set decoder_start_token_id based on your tokenizer or specific requirements
#     model.config.decoder_start_token_id = processor.tokenizer.pad_token_id  # Adjust based on your tokenizer

# Proceed with dataset loading and training setup as before
# Assuming train_df and test_df are your dataframes containing file names and texts
train_dataset = KhmerDataset(df=train_df, root_dir="dataset/data_train/", processor=processor)
test_dataset = KhmerDataset(df=test_df, root_dir="dataset/data_test/", processor=processor)

# Create dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


: 

In [34]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    evaluation_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    save_steps=10_000,
    save_total_limit=2,
    output_dir="./trocr_khmer",
    overwrite_output_dir=True,
    logging_dir='./logs',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()


/home/EmHengly/miniconda3/envs/khocr/lib/python3.10/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


ValueError: Caught ValueError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/home/EmHengly/.local/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py", line 83, in _worker
    output = module(*input, **kwargs)
  File "/home/EmHengly/.local/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1511, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/EmHengly/.local/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1520, in _call_impl
    return forward_call(*args, **kwargs)
  File "/home/EmHengly/miniconda3/envs/khocr/lib/python3.10/site-packages/transformers/models/vision_encoder_decoder/modeling_vision_encoder_decoder.py", line 610, in forward
    decoder_input_ids = shift_tokens_right(
  File "/home/EmHengly/miniconda3/envs/khocr/lib/python3.10/site-packages/transformers/models/vision_encoder_decoder/modeling_vision_encoder_decoder.py", line 44, in shift_tokens_right
    raise ValueError("

In [35]:
import os
import pandas as pd
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
from torch.utils.data import DataLoader
from PIL import Image
import torch

# Step 1: Load data from text files with custom delimiter (space or tab)
train_df = pd.read_csv("dataset/data_train.txt", delimiter='\t', names=['file_name', 'text'])
test_df = pd.read_csv("dataset/data_test.txt", delimiter='\t', names=['file_name', 'text'])

# Step 2: Specify the directory where your tokenizer files are located
tokenizer_directory = "khmer_tokenizer/khmer.model"

# Step 3: Initialize the tokenizer
tokenizer = TrOCRProcessor.from_pretrained(tokenizer_directory)

# Step 4: Define a custom dataset class to load images and tokenize text
class KhmerDataset(torch.utils.data.Dataset):
    def __init__(self, df, root_dir, processor, max_image_length=1024):
        self.df = df
        self.root_dir = root_dir
        self.processor = processor
        self.max_image_length = max_image_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_name = self.df['file_name'].iloc[idx]
        text = self.df['text'].iloc[idx]

        # Read image
        image_path = os.path.join(self.root_dir, file_name)
        image = Image.open(image_path).convert('RGB')

        # Tokenize text
        inputs = self.processor(text, image, max_length=self.max_image_length, padding="max_length", truncation=True)

        return inputs

# Step 5: Create dataset instances using the dataframes
train_dataset = KhmerDataset(df=train_df, root_dir="dataset/data_train/", processor=tokenizer)
test_dataset = KhmerDataset(df=test_df, root_dir="dataset/data_test/", processor=tokenizer)

# Step 6: Create dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)

# Step 7: Initialize the Vision Encoder-Decoder model
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')

# Step 8: Define training arguments
training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    eval_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    save_steps=10_000,
    save_total_limit=2,
    output_dir="./trocr_khmer",
    overwrite_output_dir=True,
    logging_dir='./logs',
)

# Step 9: Create the Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Step 10: Start training
trainer.train()


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 59: invalid start byte